# Figure 1 — Study design, data structure, and analytical workflow

- **FIG_ID**: `Figure 1`（编号真源：`data/figure_numbering.csv`，§4.0；output stem `Figure_1`）
- **图位**: v6 正文行 157（§2.5 末）；对应 `[FIGURE 1 PLACEHOLDER]`
- **数据依赖**: 无（纯矢量示意图；`data_layer_a = "-"`）。唯一外部溯源：archify 冻结规格
  `.archify/architecture-study-design-workflow-20261002-110640/candidate.json`（Cell 3 验哈希）
- **输出**: `out/draft/Figure_1.{png,pdf,eps}`（`FIG_DRAFT=1`，默认）/ `out/submission/Figure_1.*`
  （`FIG_DRAFT=0`）+ `out/captions/Figure_1_caption.txt` + `qc/figure_1_qc.json`
- **图面约束**（v6 行 157 + §1.6.8 流程图模板）: 横向四段（① 数据来源 ② 分析单元与时间切分
  ③ 主要分析 ④ 次级与预测分析）；**图内不放效应估计值**（无 IRR / AUC / 阈值）；
  每框 ≤2 行副文；实线箭头=数据流、虚线=预测时刻流；虚线灰框标 exploratory；
  **不列算法名**；内部代号（NB2/SPEC_P/M3）禁用
- **设计系统（§1.6，v2.3）**: NJU 主色板——紫 `#6E0F6D` = 主要关联分析，橙 `#D89000` =
  预注册阈值规格，灰 `#7A7A7A` = 探索性；浅紫/浅灰底用实色浅调（EPS 不支持透明度）；
  字号取自 `fig_common.DS`（注释 7.5 pt，图内下限 7 pt）；必须带图例说明各层级含义


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path


def _resolve_scripts_dir() -> Path:
    """定位 figure/scripts，使本册在**任何内核 cwd** 下都可运行。"""
    here = Path.cwd()
    cands = [here / "scripts", here.parent / "scripts"]
    cands += [p / "figure" / "scripts" for p in (here, *here.parents)]
    cands.append(Path(r"D:\NJU_folder\final_update\15_paper\figure\scripts"))
    for c in cands:
        if (c / "fig_common.py").exists():
            return c.resolve()
    raise RuntimeError("未找到 figure/scripts/fig_common.py：请从本仓库、figure/ 或 "
                       "figure/notebooks/ 任一目录启动内核，或检查仓库路径")


SCRIPTS_DIR = _resolve_scripts_dir()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))
print("scripts dir:", SCRIPTS_DIR)

from fig_common import *          # 样式、色板、导出、核验（唯一来源，勿在本册硬编码样式）
import matplotlib.pyplot as plt

print(f">>> Figure 1 notebook | design system {DESIGN_VERSION} | notebook API v{NOTEBOOK_API} "
      f"| 横向四段流程图 + NJU purple")
print(">>> 若本行未出现、或版本号低于上面的值，说明内核里跑的是**旧版 notebook**："
      "请在编辑器执行 Revert File（VS Code）/ 关闭标签后重新打开本文件")

# ---- 本图配置 -------------------------------------------------------------
FIG_ID = "Figure 1"
IS_DRAFT = os.environ.get("FIG_DRAFT", "1") == "1"   # 1 -> out/draft；0 -> out/submission
t0 = time.time()
meta = figure_meta(FIG_ID)
print(f"FIG_ID={FIG_ID}  stem={meta['filename_stem']}  IS_DRAFT={IS_DRAFT}")
print("data layer: none (pure vector; numbering csv data_layer_a = '-')")
print("输出目录:", FIG_OUT_DRAFT if IS_DRAFT else FIG_OUT_SUB)


In [ ]:
# ---- 核验（纯矢量图：无数据哈希；验 archify 规格哈希 + 图内代号扫描）--------
import fig01_workflow as core

# 1) manifest 在位（本图无 A 层取数文件，仅确认取数体系可用）
report = verify_figure_data(FIG_ID)

# 2) archify 冻结规格哈希（内容口径溯源；candidate.json 改动即中断）
assert core.CANDIDATE.exists(), f"缺少 archify 冻结规格：{core.CANDIDATE}"
cand_sha = sha256_file(core.CANDIDATE)
assert cand_sha == core.CANDIDATE_SHA256, (
    f"candidate.json 哈希变化（{cand_sha[:16]}…）：内容口径可能已改，请核对后再绘")
print("candidate.json sha256:", cand_sha[:16], "… (matched)")

# 3) 登记值自检（fig01_record.md §3 逐条核对口径；数值必须出现在图内文字或图注中）
REG = {
    "crashes": "5,515", "crash_days": "3,692", "gantries": "50",
    "sections": "25", "stakes": "224", "panel": "27,400", "features": "46",
}
for k, v in REG.items():
    assert v in core.CAPTION or any(v in t for t in core.ALL_TEXT), \
        f"登记值 {k}={v} 未出现在图内文字或图注中"
print("登记数值口径自检通过:", REG)
print("核验通过：", report["checked"], "| candidate provenance ok")


In [ ]:
# ---- 绘图：横向四段流程图（§2-Fig1 v6 行 157 + §1.6.8 模板）----------------
# 面板装配代码放在同图模块 fig01_workflow.build_figure() 内，
# 使命令行试绘（--draft/--submission）与本册出图共用唯一实现，避免两处拷贝漂移。
from fig01_workflow import build_figure

fig = build_figure()
print("in-figure texts:", len(core.ALL_TEXT), "lines; codename scan: passed (in-build assert)")
print("legend entries:", [t.get_text() for t in fig.legends[0].get_texts()])
fig


In [ ]:
# ---- 导出双版本 + caption + qc -------------------------------------------
written = export_figure(fig, FIG_ID, is_draft=IS_DRAFT)
print("\n".join(str(p) for p in written))

from fig01_workflow import CAPTION

cap = write_caption(FIG_ID, CAPTION,
                    title="Figure 1. Study design, data structure, and analytical workflow.")
print(cap)

qc = fig_qc_dump(FIG_ID, {
    "status": "ok", "is_draft": IS_DRAFT, "seconds": round(time.time() - t0, 1),
    "data_layer": "none (pure vector)",
    "provenance": {"candidate_json": str(core.CANDIDATE), "sha256": cand_sha},
    "palette": "NJU purple #6E0F6D (primary); orange #D89000 (threshold); grey #7A7A7A (exploratory)",
    "layout": "four horizontal stages (data sources | unit & time splits | primary | secondary & prediction)",
    "codename_scan": "passed (no NB2/SPEC_P/M3/algorithm names in-figure)",
    "outputs": [{"path": str(p), "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in written],
    "caption": str(cap),
})
print(qc)

# 投稿版凭据：只在非草稿运行时写入，避免被后续草稿重跑覆盖（发布门禁以它为准）
if not IS_DRAFT:
    rec = fig_submission_record(FIG_ID, written, cap,
                                {"data_layer": "none (pure vector)",
                                 "candidate_sha256": cand_sha})
    print("submission record:", rec)
